# Construct Demographic-Conditioned Mobility Behaviour Profile

## Overview

This notebook connects the Representative Population Layer with the Behavioural Evidence Layer of the Human Mobility Representation Framework.

The Representative Population Pool was initially constructed from the Cartesian product of the selected demographic attribute categories. As this process produced all possible combinations, logical consistency rules were applied to remove infeasible profiles, such as assigning an occupation or industry to a person who is not employed. The resulting logically constrained population pool therefore contains feasible representative combinations of demographic and socio-economic attributes.

Given one representative profile from this population pool, this notebook retrieves the corresponding trip purpose distribution based on age and gender. Each trip purpose is then associated with its temporal and travel mode distributions derived from the National Travel Survey.

The output is an unconstrained Demographic-Conditioned Mobility Behaviour
Profile that retains all supported trip purposes and their associated
temporal and travel mode probability distributions. It represents a
factorised behavioural probability space rather than a sampled or realised
journey.

Individual-level travel mode constraints and transport requirement
derivation are outside the scope of this notebook and are handled in the
subsequent framework stage.


## 1. Import Libraries

In [1]:
import json
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

## 2. Set Up File Paths

In [2]:
project_root = Path.cwd().parents[1]

population_file = (
    project_root
    / "output"
    / "demographic_profiles"
    / "representative_population.parquet"
)

behavioural_evidence_file = (
    project_root
    / "output"
    / "behavioural_evidence"
    / "nts_behavioural_evidence.json"
)

## 3. Load Input Data

In [3]:
population = pd.read_parquet(population_file)

with open(behavioural_evidence_file, "r", encoding="utf-8") as file:
    behavioural_evidence = json.load(file)

## 4. Inspect Input Data

In [4]:
print(f"Population shape: {population.shape}")
print(f"Population columns: {population.columns.tolist()}")

print(f"\nBehavioural evidence type: {type(behavioural_evidence).__name__}")
print(f"Number of trip purposes: {len(behavioural_evidence)}")

Population shape: (2869120, 14)
Population columns: ['Age', 'Gender', 'Household Type', 'Household Count', 'Economic Activity Status', 'Economic Count', 'Industry', 'Industry Count', 'Ethnic Group', 'Ethnic Count', 'Distance travelled to work', 'Travel Distance Count', 'Car Ownership', 'Car Ownership Count']

Behavioural evidence type: dict
Number of trip purposes: 14


In [5]:
display(population.head())

,Age,Gender,Household Type,Household Count,Economic Activity Status,Economic Count,Industry,Industry Count,Ethnic Group,Ethnic Count,Distance travelled to work,Travel Distance Count,Car Ownership,Car Ownership Count
0,0-4,Female,Couple household: With dependent children,20814,Does not apply,34378,Does not apply,34378,"Asian, Asian British or Asian Welsh: Bangladeshi",833,Does not apply,34378,1 car or van in household,17962
1,0-4,Female,Couple household: With dependent children,20814,Does not apply,34378,Does not apply,34378,"Asian, Asian British or Asian Welsh: Bangladeshi",833,Does not apply,34378,2 cars or vans in household,11303
2,0-4,Female,Couple household: With dependent children,20814,Does not apply,34378,Does not apply,34378,"Asian, Asian British or Asian Welsh: Bangladeshi",833,Does not apply,34378,3 or more cars or vans in household,1499
3,0-4,Female,Couple household: With dependent children,20814,Does not apply,34378,Does not apply,34378,"Asian, Asian British or Asian Welsh: Bangladeshi",833,Does not apply,34378,No cars or vans in household,5428
4,0-4,Female,Couple household: With dependent children,20814,Does not apply,34378,Does not apply,34378,"Black, Black British, Black Welsh, Caribbean or African: African",3717,Does not apply,34378,1 car or van in household,17962


In [6]:
print("Available trip purposes:")
print(list(behavioural_evidence.keys()))

Available trip purposes:
['Commuting', 'Business', 'Education', 'Escort Education', 'Shopping', 'Other Escort', 'Personal Business', 'Visiting Friends at Private Home', 'Visiting Friends Elsewhere', 'Entertainment or Public Activity', 'Sport: Participate', 'Holiday: Base', 'Day Trip', 'Just Walk']


In [7]:
example_purpose = "Commuting"

print(f"Evidence components for {example_purpose}:")
print(behavioural_evidence[example_purpose].keys())

Evidence components for Commuting:
dict_keys(['baseline_profile', 'travel_mode_profile', 'temporal_pattern_profile'])


## 5. Select and Prepare a Representative Profile

One logically feasible demographic profile is selected from the
Representative Population Pool for framework testing.

The population data retain supporting counts from the source Census tables
for validation and provenance. These counts describe different marginal or
partially conditioned categories and do not represent the joint frequency
of the complete demographic profile.

The profile is therefore separated into:

- demographic attributes, which are passed to the Behavioural Evidence
  Layer; and
- supporting Census counts, which remain within the Population Layer and
  are not treated as individual characteristics or behavioural weights.

In [8]:
representative_profile = population.iloc[0]

profile_attribute_columns = [
    "Age",
    "Gender",
    "Household Type",
    "Economic Activity Status",
    "Industry",
    "Ethnic Group",
    "Distance travelled to work",
    "Car Ownership",
]

supporting_count_columns = [
    "Household Count",
    "Economic Count",
    "Industry Count",
    "Ethnic Count",
    "Travel Distance Count",
    "Car Ownership Count",
]

profile_attributes = representative_profile[
    profile_attribute_columns
]

supporting_counts = representative_profile[
    supporting_count_columns
]

In [9]:
print(
    "Profile attributes:",
    len(profile_attributes),
)

print(
    "Supporting count fields:",
    len(supporting_counts),
)

Profile attributes: 8
Supporting count fields: 6


## 6. Prepare and Validate Purpose-Conditioned Temporal Distributions

Each unified trip purpose is linked to its corresponding temporal
distribution derived from NTS0502:

\[
P(\text{Start Time} \mid \text{Trip Purpose})
\]

Trip-purpose harmonisation was completed during the Behavioural Evidence
integration stage. The unified trip-purpose keys are therefore used directly
in this notebook. The original `Year` and `Trip purpose` fields are excluded
when extracting the 24 hourly probabilities.

Each temporal distribution is validated to ensure that:

- exactly one temporal record is available for each unified trip purpose;
- 24 hourly probabilities are available;
- every probability is between zero and one; and
- the hourly probabilities sum to one.

In [10]:
temporal_distributions = {}
temporal_validation_results = []

temporal_metadata_fields = {
    "Year",
    "Trip purpose",
}

for purpose, evidence in behavioural_evidence.items():
    temporal_records = evidence[
        "temporal_pattern_profile"
    ]

    if len(temporal_records) != 1:
        raise ValueError(
            "Expected exactly one temporal pattern for "
            f"purpose={purpose}, "
            f"but found {len(temporal_records)}."
        )

    temporal_record = temporal_records[0]

    hourly_probabilities = {
        time_period: probability
        for time_period, probability
        in temporal_record.items()
        if time_period not in temporal_metadata_fields
    }

    if len(hourly_probabilities) != 24:
        raise ValueError(
            "Expected 24 hourly probabilities for "
            f"purpose={purpose}, "
            f"but found {len(hourly_probabilities)}."
        )

    if not all(
        0 <= probability <= 1
        for probability in hourly_probabilities.values()
    ):
        raise ValueError(
            "Invalid temporal probability detected for "
            f"purpose={purpose}."
        )

    probability_sum = sum(
        hourly_probabilities.values()
    )

    if abs(probability_sum - 1.0) >= 1e-8:
        raise ValueError(
            "Temporal probabilities do not sum to 1 for "
            f"purpose={purpose}: "
            f"sum={probability_sum}."
        )

    temporal_distributions[purpose] = {
        "year": temporal_record["Year"],
        "start_time_probabilities": hourly_probabilities,
    }

    temporal_validation_results.append(
        {
            "Trip Purpose": purpose,
            "Time Periods": len(hourly_probabilities),
            "Probability Sum": probability_sum,
            "Validation": "PASS",
        }
    )

In [11]:
temporal_validation_summary = pd.DataFrame(
    temporal_validation_results
)

display(temporal_validation_summary)

print(
    "All temporal distributions validated successfully."
)

,Trip Purpose,Time Periods,Probability Sum,Validation
0,Commuting,24,1.0,PASS
1,Business,24,1.0,PASS
2,Education,24,1.0,PASS
3,Escort Education,24,1.0,PASS
4,Shopping,24,1.0,PASS
5,Other Escort,24,1.0,PASS
6,Personal Business,24,1.0,PASS
7,Visiting Friends at Private Home,24,1.0,PASS
8,Visiting Friends Elsewhere,24,1.0,PASS
9,Entertainment or Public Activity,24,1.0,PASS


All temporal distributions validated successfully.


## 7. Prepare and Validate Purpose-Conditioned Travel Mode Distributions

Each unified trip purpose is linked to its corresponding travel mode
distribution derived from NTS0409:

\[
P(\text{Travel Mode} \mid \text{Trip Purpose})
\]

Trip-purpose harmonisation was completed during the Behavioural Evidence
integration stage. The unified trip-purpose keys are therefore used directly
in this notebook. The original `Purpose` field is excluded when extracting
the 13 travel mode probabilities.

Each travel mode distribution is validated to ensure that:

- exactly one travel mode record is available for each unified trip purpose;
- 13 travel mode probabilities are available;
- every probability is between zero and one; and
- the travel mode probabilities sum to one.

In [12]:
travel_mode_distributions = {}
travel_mode_validation_results = []

travel_mode_metadata_fields = {
    "Purpose",
}

for purpose, evidence in behavioural_evidence.items():
    travel_mode_records = evidence[
        "travel_mode_profile"
    ]

    if len(travel_mode_records) != 1:
        raise ValueError(
            "Expected exactly one travel mode profile for "
            f"purpose={purpose}, "
            f"but found {len(travel_mode_records)}."
        )

    travel_mode_record = travel_mode_records[0]

    mode_probabilities = {
        mode: probability
        for mode, probability
        in travel_mode_record.items()
        if mode not in travel_mode_metadata_fields
    }

    if len(mode_probabilities) != 13:
        raise ValueError(
            "Expected 13 travel mode probabilities for "
            f"purpose={purpose}, "
            f"but found {len(mode_probabilities)}."
        )

    if not all(
        0 <= probability <= 1
        for probability in mode_probabilities.values()
    ):
        raise ValueError(
            "Invalid travel mode probability detected for "
            f"purpose={purpose}."
        )

    probability_sum = sum(
        mode_probabilities.values()
    )

    if abs(probability_sum - 1.0) >= 1e-8:
        raise ValueError(
            "Travel mode probabilities do not sum to 1 for "
            f"purpose={purpose}: "
            f"sum={probability_sum}."
        )

    travel_mode_distributions[purpose] = {
        "travel_mode_probabilities": (
            mode_probabilities
        )
    }

    travel_mode_validation_results.append(
        {
            "Trip Purpose": purpose,
            "Travel Modes": len(
                mode_probabilities
            ),
            "Probability Sum": probability_sum,
            "Validation": "PASS",
        }
    )

In [13]:
travel_mode_validation_summary = pd.DataFrame(
    travel_mode_validation_results
)

display(travel_mode_validation_summary)

print(
    "All travel mode distributions validated successfully."
)

,Trip Purpose,Travel Modes,Probability Sum,Validation
0,Commuting,13,1.0,PASS
1,Business,13,1.0,PASS
2,Education,13,1.0,PASS
3,Escort Education,13,1.0,PASS
4,Shopping,13,1.0,PASS
5,Other Escort,13,1.0,PASS
6,Personal Business,13,1.0,PASS
7,Visiting Friends at Private Home,13,1.0,PASS
8,Visiting Friends Elsewhere,13,1.0,PASS
9,Entertainment or Public Activity,13,1.0,PASS


All travel mode distributions validated successfully.


## 8. Define the Behaviour Profile Construction Function

The function accepts one feasible demographic profile and returns an
unconstrained demographic-conditioned mobility behaviour profile.

Only the eight demographic attributes are retained. Supporting Census
counts are excluded because they describe marginal source categories rather
than the joint frequency of the complete profile.

Age and gender are used to retrieve the demographic-conditioned trip purpose
distribution:

\[
P(\text{Purpose} \mid \text{Age, Gender})
\]

Each unified trip purpose is then linked to the previously validated:

\[
P(\text{Start Time} \mid \text{Purpose})
\]

and:

\[
P(\text{Travel Mode} \mid \text{Purpose})
\]

The result retains the three distributions in a factorised structure. No
behavioural sampling is performed.

In [14]:
def construct_behaviour_profile(
    representative_profile,
    behavioural_evidence,
    temporal_distributions,
    travel_mode_distributions,
):
    """Construct one unconstrained mobility behaviour profile."""

    attribute_columns = [
        "Age",
        "Gender",
        "Household Type",
        "Economic Activity Status",
        "Industry",
        "Ethnic Group",
        "Distance travelled to work",
        "Car Ownership",
    ]

    if isinstance(representative_profile, pd.Series):
        source_profile = representative_profile.to_dict()
    elif isinstance(representative_profile, dict):
        source_profile = representative_profile
    else:
        raise TypeError(
            "representative_profile must be a "
            "pandas Series or dictionary."
        )

    missing_attributes = (
        set(attribute_columns) - set(source_profile)
    )

    if missing_attributes:
        raise ValueError(
            "Missing profile attributes: "
            f"{sorted(missing_attributes)}."
        )

    profile = {
        column: source_profile[column]
        for column in attribute_columns
    }

    purpose_keys = set(behavioural_evidence)

    if (
        set(temporal_distributions) != purpose_keys
        or set(travel_mode_distributions) != purpose_keys
    ):
        raise ValueError(
            "Trip-purpose keys are inconsistent across "
            "the behavioural components."
        )

    age = profile["Age"]
    gender = profile["Gender"]
    mobility_behaviour = {}

    for purpose, evidence in behavioural_evidence.items():
        matches = [
            record
            for record in evidence["baseline_profile"]
            if record["Age"] == age
            and record["Gender"] == gender
        ]

        if len(matches) != 1:
            raise ValueError(
                "Expected one purpose probability for "
                f"{purpose}, age={age}, gender={gender}; "
                f"found {len(matches)}."
            )

        purpose_probability = matches[0][
            "Trip Purpose Probability"
        ]

        if not 0 <= purpose_probability <= 1:
            raise ValueError(
                "Invalid purpose probability for "
                f"{purpose}."
            )

        mobility_behaviour[purpose] = {
            "trip_purpose_probability": (
                purpose_probability
            ),
            "temporal_distribution": (
                temporal_distributions[purpose]
            ),
            "travel_mode_distribution": (
                travel_mode_distributions[purpose]
            ),
        }

    probability_sum = sum(
        values["trip_purpose_probability"]
        for values in mobility_behaviour.values()
    )

    if abs(probability_sum - 1.0) >= 1e-8:
        raise ValueError(
            "Trip-purpose probabilities do not sum "
            f"to 1: {probability_sum}."
        )

    return {
        "representative_profile": profile,
        "mobility_behaviour": mobility_behaviour,
    }

## 9. Test and Validate the Behaviour Profile Function

The function is applied to the selected profile. The output is validated to
confirm that it contains eight demographic attributes, fourteen unified trip
purposes, twenty-four time periods per purpose, and thirteen travel modes
per purpose.

In [15]:
test_profile = representative_profile

test_behaviour_profile = construct_behaviour_profile(
    representative_profile=test_profile,
    behavioural_evidence=behavioural_evidence,
    temporal_distributions=temporal_distributions,
    travel_mode_distributions=travel_mode_distributions,
)

print(
    "Test profile:",
    test_behaviour_profile[
        "representative_profile"
    ]["Age"],
    test_behaviour_profile[
        "representative_profile"
    ]["Gender"],
)

print(
    "Representative attributes:",
    len(
        test_behaviour_profile[
            "representative_profile"
        ]
    ),
)

print(
    "Unified trip purposes:",
    len(
        test_behaviour_profile[
            "mobility_behaviour"
        ]
    ),
)

Test profile: 0-4 Female
Representative attributes: 8
Unified trip purposes: 14


In [16]:
behaviour_summary = pd.DataFrame(
    [
        {
            "Trip Purpose": purpose,
            "Purpose Probability": values[
                "trip_purpose_probability"
            ],
            "Time Periods": len(
                values["temporal_distribution"][
                    "start_time_probabilities"
                ]
            ),
            "Travel Modes": len(
                values["travel_mode_distribution"][
                    "travel_mode_probabilities"
                ]
            ),
        }
        for purpose, values
        in test_behaviour_profile[
            "mobility_behaviour"
        ].items()
    ]
).sort_values(
    "Purpose Probability",
    ascending=False,
    ignore_index=True,
)

display(behaviour_summary)

assert len(
    test_behaviour_profile[
        "representative_profile"
    ]
) == 8

assert len(
    test_behaviour_profile[
        "mobility_behaviour"
    ]
) == 14

assert (
    behaviour_summary["Time Periods"] == 24
).all()

assert (
    behaviour_summary["Travel Modes"] == 13
).all()

assert abs(
    behaviour_summary[
        "Purpose Probability"
    ].sum()
    - 1.0
) < 1e-8

print("Behaviour profile validation: PASS")

,Trip Purpose,Purpose Probability,Time Periods,Travel Modes
0,Education,0.327535,24,13
1,Other Escort,0.142315,24,13
2,Visiting Friends at Private Home,0.106521,24,13
3,Entertainment or Public Activity,0.078185,24,13
4,Personal Business,0.075573,24,13
5,Just Walk,0.053662,24,13
6,Escort Education,0.044123,24,13
7,Visiting Friends Elsewhere,0.043625,24,13
8,Day Trip,0.041132,24,13
9,Sport: Participate,0.035980,24,13


Behaviour profile validation: PASS


In [17]:
alternative_profile = population.loc[
    (population["Age"] != test_profile["Age"])
    & (
        population["Gender"]
        != test_profile["Gender"]
    )
].iloc[0]

alternative_behaviour_profile = (
    construct_behaviour_profile(
        representative_profile=alternative_profile,
        behavioural_evidence=behavioural_evidence,
        temporal_distributions=temporal_distributions,
        travel_mode_distributions=(
            travel_mode_distributions
        ),
    )
)

print(
    "Alternative profile:",
    alternative_behaviour_profile[
        "representative_profile"
    ]["Age"],
    alternative_behaviour_profile[
        "representative_profile"
    ]["Gender"],
)

print(
    "Alternative profile validation: PASS"
)

Alternative profile: 5-11 Male
Alternative profile validation: PASS


## 10. Inspect the Final Output

The final output contains the selected demographic attributes and the
corresponding factorised mobility behaviour profile. Supporting Census
counts are not included in the behavioural output.

In [18]:
display(
    pd.Series(
        test_behaviour_profile[
            "representative_profile"
        ],
        name="Value",
    ).to_frame()
)

,Value
Age,0-4
Gender,Female
Household Type,Couple household: With dependent children
Economic Activity Status,Does not apply
Industry,Does not apply
Ethnic Group,"Asian, Asian British or Asian Welsh: Bangladeshi"
Distance travelled to work,Does not apply
Car Ownership,1 car or van in household
